# Tutorial: Decorators

1. What are decorators, and how do they work?
2. Creating simple decorators
3. Creating decorators that modify inputs
4. Creating decorators that modify outputs
5. Multiple decorators
6. Decorators that take arguments
7. Improving our decorators with ... decorators!
8. (Maybe) decorating classes, decorating with classes

# DRY -- don't repeat yourself!



In [1]:
def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(a())    
print(b())

a!

b!



In [2]:
# we need to print dashed lines above and below every printout

lines = '-' * 60 + '\n'

def a():
    return f'{lines}a!\n{lines}'

def b():
    return f'{lines}b!\n{lines}'

print(a())    
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [3]:
# option 2: instead of calling the function directly,
# let's call it via another function which will insert the lines

lines = '-' * 60 + '\n'

def with_lines(func):
    return f'{lines}{func()}{lines}'

def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(with_lines(a))
print(with_lines(b))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [4]:
# option 3: The problem with the previous option is that
# it broke existing code!

# let's instead use a nested function -- a function defined
# inside of another function!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
with_lines_a = with_lines(a)

def b():
    return f'b!\n'
with_lines_b = with_lines(b)

print(with_lines_a())
print(with_lines_b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [5]:
# option 4: Now let's avoid breaking existing code

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
a = with_lines(a)

def b():
    return f'b!\n'
b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [6]:
# option 5: use decorator syntax

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

@with_lines    # this line is 100% equivalent to line 13
def a():
    return f'a!\n'
# a = with_lines(a)

@with_lines   # this is 100% equivalent to line 18
def b():
    return f'b!\n'
# b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



# Summary of a decorator

1. A decorator is a function, which we mention with @
2. That decorator function takes an argument, a function (the "decorated" function)
3. The decorator returns a function (the inner function, often named "wrapper")
4. The returned wrapper then is assigned to the name that the decorated function originally had.

# Who cares?

A decorator allows me to hijack a function at definition time (when the outer, decorator function runs), and also at runtime (when the inner, wrapper function runs).

1. Replace a function if the permissions are wrong
2. Only allow a function to run under particular circumstances
3. Change/filter the inputs to the function
4. change/filter the outputs from the function
5. Log information about the function's running

Everything we do with decorators could be done by modifying the function itself. But that would violate the DRY rule. This allows us to apply functionality to a large number of functions without rewriting them.

In [9]:
# let's use this decorator in more places!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper(*args):
        return f'{lines}{func(*args)}{lines}'
    return wrapper

@with_lines    # this line is 100% equivalent to line 13
def a():
    return f'a!\n'

@with_lines   # this is 100% equivalent to line 18
def b():
    return f'b!\n'

print(a())
print(b())

@with_lines
def add(x, y):
    return x + y

@with_lines
def mul(x, y):
    return x * y

print(add(2, 5))    
print(add(10, 20))
print(mul(3, 7))
print(mul(10, 15))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------

------------------------------------------------------------
7------------------------------------------------------------

------------------------------------------------------------
30------------------------------------------------------------

------------------------------------------------------------
21------------------------------------------------------------

------------------------------------------------------------
150------------------------------------------------------------



# Exercise: Timing

1. Write a decorator, called `timefunc`, that checks how long it takes for a function to run. The function will run normally, with its usual arguments and returning its usual values, but we will keep track of how long it took to execute.
2. The execution time, along with the name of the function we were running, will be stored to a file called `timing.txt`.
3. You can get the current Unix time (i.e., number of seconds since Jan 1, 1970) with `time.time()`.
4. You can get the name of a function from its `__name__` attribute.
5. You can write two slow functions -- I usually write `slow_add` and `slow_mul`, and put in `time.sleep` in there, maybe even using a random number.

In [13]:
import time
import random

def timefunc(func):     # outer decorator function always takes a function
    def wrapper(*args): # wrapper always takes *args
        start_time = time.time()
        value = func(*args)
        total_time = time.time() - start_time

        with open('timing.txt', 'a') as f:
            f.write(f'{func.__name__}\t{total_time}\n')
        
        return value
    return wrapper

@timefunc
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@timefunc
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))

7
30
21
150


In [14]:
!cat timing.txt

slow_add	2.0050621032714844
slow_add	1.001225233078003
slow_mul	3.003403902053833
slow_mul	4.38690185546875e-05


# Caching

What if we set things up such that `wrapper` will check the arguments of the function that is being run. If we've never seen these arguments before, then we will run the function, storing the result in a cache. If we have seen the arguments before, we just return the result that we saw before.

This kind of caching is known as "memoization."



In [16]:
def memoize(func):
    cache = {}

    def wrapper(*args):
        if args not in cache:          # first time with these args?
            cache[args] = func(*args)  # store in our cache
        return cache[args]             # always works!
    return wrapper

@memoize
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@memoize
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))
print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))

7
30
21
150
7
30
21
150


# Exercise: `once_per_minute`

Write a decorator, `once_per_minute`, that when applied to a function, raises an exception (`CalledTooSoon`) if the function is invoked more than once in a 60-second period.

In [22]:
class CalledTooSoonError(Exception):
    pass

def once_per_minute(func):
    last_called_at = 0

    def wrapper(*args):
        nonlocal last_called_at
        current_time = time.time()

        remaining_time = current_time - last_called_at
        if remaining_time < 60:
            raise CalledTooSoonError(f'Too soon; wait {remaining_time} more')

        last_called_at = current_time
        return func(*args)
    return wrapper

@once_per_minute
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@once_per_minute
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_mul(3, 7))
print(slow_add(10, 20))
print(slow_mul(10, 15))


7
21


CalledTooSoonError: Too soon; wait 5.009716987609863 more

In [21]:
x = 100    # this takes the value (100) and assigns it to the variable

In [23]:
d = {}    # this assigns a dict to d
d['a'] = 100   # this actually is rewritten to

# d.__setitem__('a', 100)

In [25]:
slow_add.__code__.co_varnames

('args', 'current_time', 'remaining_time')

In [27]:
slow_add.__code__.co_freevars

('func', 'last_called_at')

In [31]:
import dis

In [33]:
dis.dis(slow_add)

  --           COPY_FREE_VARS           2

   7           RESUME                   0

   9           LOAD_GLOBAL              0 (time)
               LOAD_ATTR                1 (time + NULL|self)
               CALL                     0
               STORE_FAST               1 (current_time)

  11           LOAD_FAST_BORROW         1 (current_time)
               LOAD_DEREF               4 (last_called_at)
               BINARY_OP               10 (-)
               STORE_FAST               2 (remaining_time)

  12           LOAD_FAST_BORROW         2 (remaining_time)
               LOAD_SMALL_INT          60
               COMPARE_OP              18 (bool(<))
               POP_JUMP_IF_FALSE       16 (to L1)
               NOT_TAKEN

  13           LOAD_GLOBAL              3 (CalledTooSoonError + NULL)
               LOAD_CONST               1 ('Too soon; wait ')
               LOAD_FAST_BORROW         2 (remaining_time)
               FORMAT_SIMPLE
               LOAD_CONST        

# Next up

1. checking/modifying/filter arguments
2. checking/modifying/filtering return values
3. Nested decorators
3. Better decorators
4. Decorators with arguments

# What is a decorator?

- A function
- That takes a function
- And returns a function

Because of this, the decorator can:
- Hijack the creation of the function (the outer, decorator function) and do whatever it wants
- Hijack the execution of the function (the inner, wrapper function) and do whatever it wants

This means that `wrapper` can examine and modify the inputs (arguments) that we get, and also examine/modify the outputs (the return values).

In [35]:
def mysum(*args):
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 20, 35, 40, 50)    

155

In [37]:
# I'm going to write a decorator called only_odds
# that will filter through the arguments, and only keep the odd numbers

def only_odds(func):
    def wrapper(*args):
        odd_numbers = []

        for one_arg in args:
            if one_arg % 2:   # if the remainder is 1, then it's odd!
                odd_numbers.append(one_arg)
        
        value = func(*odd_numbers)
        return value
    return wrapper

@only_odds
def mysum(*args):
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 20, 35, 40, 50)        

35

In [39]:
# I'm going to write a decorator called only_odds
# that will filter through the arguments, and only keep the odd numbers

def only_odds(func):
    def wrapper(*args):
        return func(*(one_arg
                    for one_arg in args
                    if one_arg % 2))
    return wrapper

@only_odds
def mysum(*args):
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 20, 35, 40, 50)        

35

# Exercise: `only_ints`

Write a decorator `only_ints` that filters the arguments to a function, and ensures that all of the values passed are integers.

You should be able to call 

```python
mysum(10, 15, 'hello', 20, 30, 'goodbye')
```

(if you've decorated `mysum` with `only_ints`), and you'll get 75 and no complaints.


In [43]:
def only_ints(func):
    def wrapper(*args):
        """Invokes func on all int/float arguments passed to the caller"""
        return func(*(one_arg
                     for one_arg in args
                     if isinstance(one_arg, (int, float))))
    return wrapper        

@only_ints
def mysum(*args):
    """Returns the sum of all arguments, which are assumed to be numeric"""
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 15, 'hello', 20, 30, 'goodbye')

75

We know that after decorating `mysum`, it is now really referring to `wrapper`, the instance of `wrapper` that we got from invoking `only_ints` on `mysum`. 

What happens if I try to get documentation on that function?

In [44]:
help(mysum)

Help on function wrapper in module __main__:

wrapper(*args)
    Invokes func on all int/float arguments passed to the caller



In [45]:
mysum.__name__

'wrapper'

In [46]:
mysum.__code__.co_varnames

('args',)

In [49]:
# let's fix the help stuff!

def only_ints(func):
    def wrapper(*args):
        """Invokes func on all int/float arguments passed to the caller"""
        return func(*(one_arg
                     for one_arg in args
                     if isinstance(one_arg, (int, float))))

    # fix the docstring
    wrapper.__doc__ = func.__doc__
    wrapper.__name__ = func.__name__
    return wrapper        

@only_ints
def mysum(*args):
    """Returns the sum of all arguments, which are assumed to be numeric"""
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 15, 'hello', 20, 30, 'goodbye')

75

In [50]:
help(mysum)

Help on function mysum in module __main__:

mysum(*args)
    Returns the sum of all arguments, which are assumed to be numeric



In [51]:
# much better

import functools

def only_ints(func):

    @functools.wraps(func)
    def wrapper(*args):
        """Invokes func on all int/float arguments passed to the caller"""
        return func(*(one_arg
                     for one_arg in args
                     if isinstance(one_arg, (int, float))))

    return wrapper        

@only_ints
def mysum(*args):
    """Returns the sum of all arguments, which are assumed to be numeric"""
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 15, 'hello', 20, 30, 'goodbye')

75

In [52]:
help(mysum)

Help on function mysum in module __main__:

mysum(*args)
    Returns the sum of all arguments, which are assumed to be numeric



# Changing the outputs

Just as we can change the inputs to a function (the arguments), we can also change the outputs.

When you return something, you then change the return value that you got from the original function. This can involve logging, changing the type, or even performing a transformation.

# Exercise: `only_return_ints`

Write a decorator that examines the return value from the function. You can assume (hope!) that the return value is a list of values, and you want to keep only those that are integers.

In [59]:
def only_return_ints(func):
    def wrapper(*args):
        return list(one_result
                for one_result in func(*args)
                if isinstance(one_result, int))
    return wrapper

@only_return_ints
def return_squares(*args):
    return [one_arg * one_arg
            for one_arg in args] + ['a', 'b', 'c']

return_squares(10, 20, 30)    

[100, 400, 900]

In [60]:
# apply the same sequence type that we got

def only_return_ints(func):
    def wrapper(*args):
        t = type(args)
        return t(one_result
                for one_result in func(*args)
                if isinstance(one_result, int))
    return wrapper

@only_return_ints
def return_squares(*args):
    return [one_arg * one_arg
            for one_arg in args] + ['a', 'b', 'c']

return_squares(10, 20, 30)    

(100, 400, 900)

In [63]:
# what if I want to apply more than one decorator?
# we've already created (a) timefunc and (b) once_per_minute

@once_per_minute
@timefunc
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y
# slow_add = timefunc(slow_add)
# slow_add = once_per_minute(slow_add)

@once_per_minute
@timefunc
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 3))    
print(slow_mul(3, 4))
print(slow_add(20, 30))    
print(slow_mul(30, 40))

5
12


CalledTooSoonError: Too soon; wait 2.0060007572174072 more

# Exercise: Multiple decorators

1. Apply two of our existing decorators to functions.
2. First modify the decorators to tell you when (a) they're being applied at definition time and (b) when they're being applied at runtime.

In [64]:
def only_ints(func):
    print(f'decorating {func.__name__} with only_ints')
    @functools.wraps(func)
    def wrapper(*args):
        """Invokes func on all int/float arguments passed to the caller"""
        print(f'in only_ints wrapper for {func.__name__} with args {args}')
    
        return func(*(one_arg
                     for one_arg in args
                     if isinstance(one_arg, (int, float))))

    return wrapper        

def timefunc(func):     # outer decorator function always takes a function
    print(f'decorating {func.__name__} with timefunc')
    def wrapper(*args): # wrapper always takes *args
        print(f'in timefunc wrapper for {func.__name__} with args {args}')
        start_time = time.time()
        value = func(*args)
        total_time = time.time() - start_time

        with open('timing.txt', 'a') as f:
            f.write(f'{func.__name__}\t{total_time}\n')
        
        return value
    return wrapper

@timefunc
@only_ints
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@timefunc
@only_ints
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 3))    
print(slow_mul(3, 4))
print(slow_add(20, 30))    
print(slow_mul(30, 40))

decorating slow_add with only_ints
decorating slow_add with timefunc
decorating slow_mul with only_ints
decorating slow_mul with timefunc
in timefunc wrapper for slow_add with args (2, 3)
in only_ints wrapper for slow_add with args (2, 3)
5
in timefunc wrapper for slow_mul with args (3, 4)
in only_ints wrapper for slow_mul with args (3, 4)
12
in timefunc wrapper for slow_add with args (20, 30)
in only_ints wrapper for slow_add with args (20, 30)
50
in timefunc wrapper for slow_mul with args (30, 40)
in only_ints wrapper for slow_mul with args (30, 40)
1200


# Decorators with arguments

Sometimes, we'll want to write a decorator that takes an argument. How would that look? How would that work?

In [ ]:
def only_ints(func):
    @functools.wraps(func)
    def wrapper(*args):
        return func(*(one_arg
                     for one_arg in args
                     if isinstance(one_arg, int)))

    return wrapper        

@only_ints
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

slow_add(2, 3)    
